# Seminar 01: PyTorch Tensors, Autograd, and Your First MLP

**Student Version**

Goals for today:
- Get comfortable with **tensors** and **shapes**
- Understand **autograd** and gradient accumulation
- Implement a **linear model** and do manual gradient descent
- See why **non-linearity** matters (linear vs MLP on a non-linear dataset)
- Train a tiny MLP in PyTorch

> Tip: Keep an eye on shapes. Most DL bugs are shape bugs.


## 0. Setup

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

### Helper functions
We will use a small 2D toy dataset and a couple of plotting helpers.

In [ ]:
def make_rings(n=800, noise=0.12, r_inner=0.6, r_outer=1.2):
    """
    Binary classification dataset:
    class 0: points near an inner ring
    class 1: points near an outer ring
    """
    n0 = n // 2
    n1 = n - n0

    angles0 = np.random.rand(n0) * 2 * np.pi
    angles1 = np.random.rand(n1) * 2 * np.pi

    r0 = r_inner + np.random.randn(n0) * noise
    r1 = r_outer + np.random.randn(n1) * noise

    x0 = np.stack([r0 * np.cos(angles0), r0 * np.sin(angles0)], axis=1)
    x1 = np.stack([r1 * np.cos(angles1), r1 * np.sin(angles1)], axis=1)

    X = np.concatenate([x0, x1], axis=0).astype(np.float32)
    y = np.concatenate([np.zeros(n0), np.ones(n1)], axis=0).astype(np.float32)

    idx = np.random.permutation(n)
    return X[idx], y[idx]

def plot_points(X, y, title='Dataset'):
    X = np.asarray(X)
    y = np.asarray(y)
    plt.figure()
    plt.scatter(X[:, 0], X[:, 1], c=y, s=12)
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

@torch.no_grad()
def plot_decision_boundary(model, X, y, title='Decision boundary', grid_steps=220):
    model.eval()
    X_np = np.asarray(X, dtype=np.float32)
    y_np = np.asarray(y)
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_steps),
        np.linspace(y_min, y_max, grid_steps),
    )
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1).astype(np.float32)
    grid_t = torch.from_numpy(grid).to(next(model.parameters()).device)

    logits = model(grid_t).reshape(-1)
    probs = torch.sigmoid(logits).cpu().numpy()
    zz = probs.reshape(xx.shape)

    plt.figure()
    plt.contourf(xx, yy, zz, levels=25)
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=12, edgecolors='k', linewidths=0.2)
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

def accuracy_from_logits(logits, y_true):
    logits = logits.reshape(-1)
    preds = (torch.sigmoid(logits) >= 0.5).float()
    return (preds == y_true).float().mean().item()

## 1. Warm-up: tensors and shapes
In this section you will practice creating tensors and reshaping them.

### Exercise 1.1
Create:
- `x` with shape `(3, 4)` filled with random numbers from standard normal
- `b` with shape `(4,)` filled with zeros
- `y = x + b` using broadcasting (should have shape `(3, 4)`)

Then create `z` by reshaping `y` into shape `(2, 6)`.

Finally, create `w` by adding a batch dimension to `b` so that it becomes shape `(1, 4)`.


In [ ]:
# Exercise 1.1

# TODO: create x, b
# x: torch.randn(3, 4)
# b: torch.zeros(4)

# TODO: broadcasting add
# y = x + b

# TODO: reshape y into z with shape (2, 6)

# TODO: add batch dimension to b -> w with shape (1, 4)

raise NotImplementedError("Fill in Exercise 1.1")

### Checks (Exercise 1.1)

In [ ]:
# Checks
assert x.shape == (3, 4)
assert b.shape == (4,)
assert y.shape == (3, 4)
assert z.shape == (2, 6)
assert w.shape == (1, 4)
print('Exercise 1.1 passed.')

## 2. Autograd essentials
Autograd tracks operations on tensors with `requires_grad=True` and can compute gradients via `.backward()`.

### Exercise 2.1
Given vectors `a` and `c`, define:

$L = \sum (a \odot c)^2$

Compute `dL/da` using autograd.

Then demonstrate **gradient accumulation** by calling `.backward()` twice without zeroing gradients.


In [ ]:
# Exercise 2.1

a = torch.randn(5, requires_grad=True)
c = torch.randn(5)

# TODO: compute L = sum((a * c)**2)

# TODO: call backward, store grad1 = a.grad.clone()

# TODO: call backward again WITHOUT zeroing grads, store grad2 = a.grad.clone()

raise NotImplementedError("Fill in Exercise 2.1")

### Checks (Exercise 2.1)

In [ ]:
# Checks
assert grad1.shape == a.shape
assert torch.allclose(grad2, 2 * grad1, atol=1e-5, rtol=1e-4)
print('Exercise 2.1 passed.')

## 3. Linear model and manual gradient descent
We will implement:
- linear model: `y_hat = X @ w + b`
- MSE loss
- manual gradient descent updates

### Exercise 3.1
Generate synthetic regression data and fit a linear model with manual updates.

Tasks:
1) Create `X` of shape `(N, 2)` and true parameters `w_true`, `b_true`
2) Generate `y = X @ w_true + b_true + noise`
3) Initialize `w`, `b` with `requires_grad=True`
4) Implement training loop for 200 steps:
   - forward pass
   - MSE loss
   - backward
   - update params using learning rate `lr`
   - zero grads

Plot the loss curve.


In [ ]:
# Data for Exercise 3.1
seed_everything(7)

N = 400
X = torch.randn(N, 2)
w_true = torch.tensor([2.0, -3.0])
b_true = torch.tensor(0.7)

noise = 0.2 * torch.randn(N)
y = X @ w_true + b_true + noise
y = y.unsqueeze(1)  # shape (N, 1)

X.shape, y.shape

In [ ]:
# Exercise 3.1

# TODO: initialize parameters
# w: shape (2, 1)
# b: shape (1,)
# both require gradients

lr = 0.05
steps = 200
losses = []

for step in range(steps):
    # TODO: forward pass y_hat = X @ w + b
    # TODO: MSE loss
    # TODO: backward
    # TODO: update w and b (with torch.no_grad())
    # TODO: zero grads
    pass

# TODO: plot losses

raise NotImplementedError("Fill in Exercise 3.1")

### Checks (Exercise 3.1)

In [ ]:
# Checks
assert isinstance(losses, list) and len(losses) > 10
assert math.isfinite(losses[-1])
assert losses[-1] < 0.3 * losses[0]
print('Exercise 3.1 passed.')

## 4. Non-linearity: linear model vs MLP on a non-linear dataset
We will use a ring dataset that is **not** linearly separable.

### Exercise 4.1
Train a **linear classifier** and show it struggles.

We will use:
- model: `nn.Linear(in_features, out_features)`
- loss: `nn.BCEWithLogitsLoss()`
- optimizer: SGD


In [ ]:
# Data for Exercise 4.1
seed_everything(123)
X_np, y_np = make_rings(n=1000, noise=0.10, r_inner=0.6, r_outer=1.2)
plot_points(X_np, y_np, title='Rings dataset (non-linear)')

In [ ]:
# Exercise 4.1

X_t = torch.from_numpy(X_np).to(device)
y_t = torch.from_numpy(y_np).to(device)

# TODO: define linear model on device
# model = nn.Linear(...).to(device)

# TODO: define loss and optimizer
# loss_fn = nn.BCEWithLogitsLoss()
# opt = torch.optim.SGD(model.parameters(), lr=...)

epochs = 200
loss_hist = []

for epoch in range(epochs):
    # TODO: forward
    # TODO: compute loss
    # TODO: backward
    # TODO: opt.step, opt.zero_grad
    pass

# TODO: compute train accuracy
# TODO: plot decision boundary

raise NotImplementedError("Fill in Exercise 4.1")

### Check (Exercise 4.1)
If your linear model reaches ~0.95+ accuracy, something is off (dataset or training).

In [ ]:
print('If you see very high accuracy here, double-check the dataset generation and labels.')

## 5. First tiny MLP in PyTorch
Now we add a hidden layer with a non-linearity and train again.

### Exercise 5.1
Build and train:
`Linear(2 -> 16) -> ReLU -> Linear(16 -> 1)`

Target: significantly better accuracy than the linear model.


In [ ]:
# Exercise 5.1

X_t = torch.from_numpy(X_np).to(device)
y_t = torch.from_numpy(y_np).to(device)

# TODO: define MLP model
# model = nn.Sequential(
#     ...
# ).to(device)

# TODO: loss and optimizer (try Adam)
# loss_fn = nn.BCEWithLogitsLoss()
# opt = torch.optim.Adam(...)

epochs = 300
loss_hist = []

for epoch in range(epochs):
    # TODO: forward -> logits
    # TODO: loss
    # TODO: backward
    # TODO: step + zero_grad
    pass

# TODO: compute accuracy
# TODO: plot loss and decision boundary

raise NotImplementedError("Fill in Exercise 5.1")

### Checks (Exercise 5.1)

In [ ]:
# Checks
assert len(loss_hist) > 50
assert math.isfinite(loss_hist[-1])
assert loss_hist[-1] < 0.7 * loss_hist[0]
print('Exercise 5.1 basic checks passed.')

## 6. Debugging mini checklist
Common issues you will hit during the course:

1) **Gradients accumulate** if you forget `zero_grad()`
2) Using `Sigmoid` + `BCELoss` vs `BCEWithLogitsLoss`
   - Prefer `BCEWithLogitsLoss` and pass **raw logits** (no sigmoid)
3) Learning rate too large: loss becomes `nan` or explodes
4) Wrong shapes: targets for BCE should match logits shape

### Optional mini task
Try to break your training on purpose:
- Use a learning rate of `1.0`
- Or forget `opt.zero_grad()`
Then fix it and write 2 sentences on what happened.


## 7. Wrap-up questions
Answer briefly in markdown:

1) Why does stacking two linear layers without activation still produce a linear function?
2) What does `.backward()` compute and where do gradients appear?
3) Why is `BCEWithLogitsLoss` often preferred over `Sigmoid` + `BCELoss`?


### Stretch goal
Try different hidden sizes (4, 16, 64) and compare:
- convergence speed
- final accuracy

Write your observations.
